# Test Drug Embeddings

This notebook allows you to:
1. Load a CSV of test drugs (by name, SMILES, or DrugBank ID)
2. Fetch required information via APIs (PubChem)
3. Match drugs to preprocessed Madrigal data
4. Generate embeddings using trained model
5. Visualize embeddings in 2D/3D and analyze distances

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

# Add project root to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

from madrigal.utils import DATA_DIR, BASE_DIR
from madrigal.models.models import NovelDDIEncoder, NovelDDIMultilabel
from madrigal.utils import to_device
from madrigal.evaluate.eval_utils import get_evaluate_masks

# Import index modules
from index.drug_lookup import DrugLookup
from index.data_retriever import DataRetriever

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

## 1. Load Test Drug CSV

Your CSV should have one of these columns:
- `drug_name`: Drug names (e.g., "Aspirin", "Metformin")
- `smiles`: SMILES strings
- `drugbank_id`: DrugBank IDs (e.g., "DB00945")

In [ ]:
# Option 1: Load from CSV file
# test_drugs = pd.read_csv('your_test_drugs.csv')

# Option 2: Create example test drugs manually
test_drugs = pd.DataFrame({
    'drug_name': [
        'Aspirin',
        'Metformin',
        'Ibuprofen',
        'Acetaminophen',
        'Omeprazole',
        'Lisinopril',
        'Atorvastatin',
        'Amlodipine',
        'Metoprolol',
        'Losartan'
    ]
})

print(f"Loaded {len(test_drugs)} test drugs")
test_drugs.head(10)

## 2. Fetch Drug Information via API

Use PubChem API to get canonical SMILES for each drug.

In [ ]:
# Initialize drug lookup
lookup = DrugLookup(cache_results=True)

# Determine input type
if 'drug_name' in test_drugs.columns:
    id_type = 'name'
    identifiers = test_drugs['drug_name'].tolist()
elif 'smiles' in test_drugs.columns:
    id_type = 'smiles'
    identifiers = test_drugs['smiles'].tolist()
elif 'drugbank_id' in test_drugs.columns:
    id_type = 'name'  # Will search by name/ID
    identifiers = test_drugs['drugbank_id'].tolist()
else:
    raise ValueError("CSV must have 'drug_name', 'smiles', or 'drugbank_id' column")

print(f"Looking up {len(identifiers)} drugs by {id_type}...")
drug_info = lookup.lookup_drugs_batch(identifiers, id_type=id_type, delay=0.3)

print(f"\nFound {(drug_info['status'] == 'found').sum()}/{len(drug_info)} drugs")
drug_info

## 3. Match Drugs to Madrigal Preprocessed Data

Check which drugs exist in the preprocessed data and what modalities are available.

In [ ]:
# Initialize data retriever
retriever = DataRetriever(data_dir=DATA_DIR, base_dir=BASE_DIR)

# Show summary of available drugs
print("Available drugs in Madrigal dataset:")
retriever.get_available_drugs_summary()

In [ ]:
# Match test drugs to preprocessed data
found_smiles = drug_info[drug_info['status'] == 'found']['canonical_smiles'].tolist()
match_results = retriever.find_drugs_batch(found_smiles)

print(f"\nMatched {match_results['found'].sum()}/{len(match_results)} drugs to preprocessed data")
match_results

In [ ]:
# Get indices of matched drugs
matched_drugs = match_results[match_results['found']]
# FIXED: Convert to integers (pandas stores as float due to NaN values)
drug_indices = [int(x) for x in matched_drugs['index'].tolist()]

print(f"Drug indices for embedding generation: {drug_indices}")

# Show modality availability
if len(drug_indices) > 0:
    modality_avail = retriever.get_modality_availability(drug_indices)
    print("\nModality availability:")
    display(modality_avail)

## 4. Generate Embeddings Using Trained Model

Load a trained Madrigal checkpoint and generate embeddings for matched drugs.

In [ ]:
# Configuration
data_source = 'DrugBank'
split_method = 'split_by_pairs'
eval_type = 'full_full'
finetune_mode = 'str_str+random_sample'

# Set checkpoint - can be directory or .pt file path
checkpoint_name = 'revived-aardvark-8'  # Example checkpoint name
checkpoint_path = BASE_DIR + f'model_output/{data_source}/{split_method}/{checkpoint_name}/'

# FIXED: Check if checkpoint exists
checkpoint_file_check = os.path.join(checkpoint_path, 'best_model.pt') if not checkpoint_path.endswith('.pt') else checkpoint_path
if os.path.exists(checkpoint_file_check):
    print(f"Checkpoint found: {checkpoint_file_check}")
else:
    print(f"WARNING: Checkpoint not found at {checkpoint_file_check}")
    print("Please update 'checkpoint_name' or 'checkpoint_path' to point to your trained model.")

In [ ]:
# Load data for matched drugs
from madrigal.evaluate.predict import get_data_for_analysis_all_drugs

# FIXED: Ensure checkpoint_path points to the actual .pt file
if checkpoint_path and not checkpoint_path.endswith('.pt'):
    checkpoint_file = os.path.join(checkpoint_path, 'best_model.pt')
else:
    checkpoint_file = checkpoint_path

if checkpoint_file and os.path.exists(checkpoint_file) and len(drug_indices) > 0:
    _, _, batch, label_map = get_data_for_analysis_all_drugs(
        data_source=data_source,
        kg_encoder='hgt',
        split_method=split_method,
        repeat=None,
        path_base=DATA_DIR,
        checkpoint=checkpoint_file,
        first_num_drugs=max(drug_indices) + 1,  # Load enough drugs
        add_specific_drugs=None
    )
    print(f"Loaded batch with {batch['head']['drugs'].shape[0]} drugs")
else:
    print("Skipping data loading - set checkpoint_path first or checkpoint file not found")
    if checkpoint_file:
        print(f"  Checkpoint path: {checkpoint_file}")
        print(f"  Exists: {os.path.exists(checkpoint_file) if checkpoint_file else False}")

In [ ]:
# Load model and generate embeddings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# FIXED: Use checkpoint_file from previous cell
if 'checkpoint_file' in dir() and checkpoint_file and os.path.exists(checkpoint_file):
    # Load checkpoint
    checkpoint = torch.load(checkpoint_file, map_location="cpu")
    
    # Build model
    encoder = NovelDDIEncoder(**checkpoint['encoder_configs'])
    model = NovelDDIMultilabel(encoder, **checkpoint['model_configs'])
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    model.to(device)
    
    print(f"Loaded model from epoch {checkpoint.get('epoch', 'unknown')}")
else:
    print("Checkpoint not found - skipping model loading")

In [ ]:
# Generate embeddings
if 'model' in dir() and 'batch' in dir():
    batch_head = to_device(batch['head'], device)
    batch_kg = to_device(batch['kg'], device)
    head_masks_base = batch['head']['masks']
    tail_masks_base = batch['tail']['masks']
    
    masks_head, masks_tail = get_evaluate_masks(head_masks_base, tail_masks_base, eval_type, finetune_mode, device)
    
    # Extract components
    head_drugs = batch_head['drugs']
    head_mol_strs = batch_head['strs']
    head_cv = batch_head['cv']
    head_tx = batch_head['tx']
    
    # Generate embeddings
    with torch.no_grad():
        z_all = model.encoder(head_drugs, masks_head, head_mol_strs, batch_kg, head_cv, head_tx)
        if model.normalize:
            z_all = F.normalize(z_all)
    
    # Extract embeddings for our test drugs
    z_test = z_all[drug_indices].detach().cpu().numpy()
    
    print(f"Generated embeddings shape: {z_test.shape}")
    print(f"Embedding dimension: {z_test.shape[1]}")
else:
    print("Model or batch not loaded - cannot generate embeddings")
    # Create dummy embeddings for visualization demo
    z_test = np.random.randn(len(drug_indices) if drug_indices else 10, 128)
    print(f"Using random embeddings for demo: {z_test.shape}")

## 5. Visualize Embeddings (2D and 3D)

Use dimensionality reduction (UMAP, t-SNE, PCA) to visualize embeddings and analyze distances.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from IPython.display import Image, display

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP not installed. Using t-SNE instead. Install with: pip install umap-learn")

In [ ]:
# Get drug names for labeling
if 'matched_drugs' in dir() and len(matched_drugs) > 0:
    drug_names = drug_info[drug_info['status'] == 'found']['identifier'].values[:len(z_test)]
else:
    drug_names = [f"Drug_{i}" for i in range(len(z_test))]

print(f"Drug names: {list(drug_names)}")

In [ ]:
# 2D Visualization using PCA
pca_2d = PCA(n_components=2)
z_pca_2d = pca_2d.fit_transform(z_test)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(z_pca_2d[:, 0], z_pca_2d[:, 1], s=100, c=range(len(z_test)), cmap='tab10')

# Add labels
for i, name in enumerate(drug_names):
    ax.annotate(name, (z_pca_2d[i, 0], z_pca_2d[i, 1]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Drug Embeddings - PCA 2D Projection')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('embeddings_pca_2d.png', dpi=150, bbox_inches='tight')
plt.close()
display(Image('embeddings_pca_2d.png'))

In [ ]:
# 2D Visualization using t-SNE (better for local structure)
if len(z_test) >= 5:  # t-SNE needs enough samples
    perplexity = min(30, len(z_test) - 1)
    tsne_2d = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    z_tsne_2d = tsne_2d.fit_transform(z_test)

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(z_tsne_2d[:, 0], z_tsne_2d[:, 1], s=100, c=range(len(z_test)), cmap='tab10')

    for i, name in enumerate(drug_names):
        ax.annotate(name, (z_tsne_2d[i, 0], z_tsne_2d[i, 1]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)

    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.set_title('Drug Embeddings - t-SNE 2D Projection')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('embeddings_tsne_2d.png', dpi=150, bbox_inches='tight')
    plt.close()
    display(Image('embeddings_tsne_2d.png'))
else:
    print("Not enough samples for t-SNE (need >= 5)")

In [ ]:
# 3D Visualization using PCA
from mpl_toolkits.mplot3d import Axes3D

pca_3d = PCA(n_components=3)
z_pca_3d = pca_3d.fit_transform(z_test)

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(z_pca_3d[:, 0], z_pca_3d[:, 1], z_pca_3d[:, 2], 
                     s=100, c=range(len(z_test)), cmap='tab10')

# Add labels
for i, name in enumerate(drug_names):
    ax.text(z_pca_3d[i, 0], z_pca_3d[i, 1], z_pca_3d[i, 2], name, fontsize=8)

ax.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]:.1%})')
ax.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]:.1%})')
ax.set_title('Drug Embeddings - PCA 3D Projection')

plt.tight_layout()
plt.savefig('embeddings_pca_3d.png', dpi=150, bbox_inches='tight')
plt.close()
display(Image('embeddings_pca_3d.png'))

## 6. Analyze Embedding Distances

Compute pairwise distances to evaluate embedding quality.

In [ ]:
# Compute pairwise distances
# Euclidean distance
euclidean_dist = squareform(pdist(z_test, metric='euclidean'))

# Cosine distance (1 - cosine similarity)
cosine_dist = squareform(pdist(z_test, metric='cosine'))

# Create distance DataFrames
dist_df_euclidean = pd.DataFrame(euclidean_dist, index=drug_names, columns=drug_names)
dist_df_cosine = pd.DataFrame(cosine_dist, index=drug_names, columns=drug_names)

print("Euclidean Distance Matrix:")
display(dist_df_euclidean.round(3))

In [ ]:
print("Cosine Distance Matrix (0 = identical, 2 = opposite):")
display(dist_df_cosine.round(3))

In [ ]:
# Visualize distance heatmap
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Euclidean distance heatmap
sns.heatmap(dist_df_euclidean, annot=True, fmt='.2f', cmap='viridis', ax=axes[0])
axes[0].set_title('Euclidean Distance')

# Cosine distance heatmap
sns.heatmap(dist_df_cosine, annot=True, fmt='.2f', cmap='viridis', ax=axes[1])
axes[1].set_title('Cosine Distance')

plt.tight_layout()
plt.savefig('embedding_distances.png', dpi=150, bbox_inches='tight')
plt.close()
display(Image('embedding_distances.png'))

In [ ]:
# Find most similar drug pairs
def get_top_similar_pairs(dist_matrix, drug_names, k=5):
    """Get top-k most similar drug pairs."""
    pairs = []
    n = len(drug_names)
    for i in range(n):
        for j in range(i+1, n):
            pairs.append({
                'drug_1': drug_names[i],
                'drug_2': drug_names[j],
                'distance': dist_matrix[i, j]
            })
    
    pairs_df = pd.DataFrame(pairs).sort_values('distance')
    return pairs_df.head(k)

print("Top 5 Most Similar Drug Pairs (by Cosine Distance):")
get_top_similar_pairs(cosine_dist, list(drug_names), k=5)

In [ ]:
# Distance statistics
print("\nDistance Statistics:")
print(f"Euclidean - Mean: {euclidean_dist[np.triu_indices(len(z_test), k=1)].mean():.3f}, "
      f"Std: {euclidean_dist[np.triu_indices(len(z_test), k=1)].std():.3f}")
print(f"Cosine    - Mean: {cosine_dist[np.triu_indices(len(z_test), k=1)].mean():.3f}, "
      f"Std: {cosine_dist[np.triu_indices(len(z_test), k=1)].std():.3f}")

## 7. Save Results

In [ ]:
# Save embeddings
output_dir = 'embedding_results'
os.makedirs(output_dir, exist_ok=True)

# Save embeddings as numpy
np.save(f'{output_dir}/test_drug_embeddings.npy', z_test)

# Save embeddings with drug names as DataFrame
embeddings_df = pd.DataFrame(z_test, index=drug_names)
embeddings_df.to_csv(f'{output_dir}/test_drug_embeddings.csv')

# Save distance matrices
dist_df_euclidean.to_csv(f'{output_dir}/euclidean_distances.csv')
dist_df_cosine.to_csv(f'{output_dir}/cosine_distances.csv')

print(f"Results saved to '{output_dir}/' directory")
print(f"  - test_drug_embeddings.npy")
print(f"  - test_drug_embeddings.csv")
print(f"  - euclidean_distances.csv")
print(f"  - cosine_distances.csv")

## 8. Interactive 3D Plot (Optional)

Use Plotly for interactive 3D visualization.

In [ ]:
try:
    import plotly.express as px
    import plotly.graph_objects as go
    
    # Create interactive 3D scatter plot
    fig = px.scatter_3d(
        x=z_pca_3d[:, 0],
        y=z_pca_3d[:, 1],
        z=z_pca_3d[:, 2],
        text=drug_names,
        title='Drug Embeddings - Interactive 3D PCA'
    )
    
    fig.update_traces(marker=dict(size=8), textposition='top center')
    fig.update_layout(
        scene=dict(
            xaxis_title=f'PC1 ({pca_3d.explained_variance_ratio_[0]:.1%})',
            yaxis_title=f'PC2 ({pca_3d.explained_variance_ratio_[1]:.1%})',
            zaxis_title=f'PC3 ({pca_3d.explained_variance_ratio_[2]:.1%})'
        )
    )
    
    # Save as HTML for interactive viewing
    fig.write_html(f'{output_dir}/interactive_3d_plot.html')
    print(f"Interactive plot saved to '{output_dir}/interactive_3d_plot.html'")
    
    # Display in notebook
    fig.show()
    
except ImportError:
    print("Plotly not installed. Install with: pip install plotly")
    print("Skipping interactive 3D plot.")